# OpenAI and Gradio UI

This notebook builds a simple text-to-image design generator for banner and poster concepts. It uses the OpenAI API for image generation and Gradio for the web interface.

## Project Goal

The app takes a text prompt and generates a bespoke design image that could be used for digital marketing campaigns such as Netflix promotional banners and posters.

In [ ]:
# Uncomment this cell if you need to install dependencies in a notebook environment.
# !pip install openai gradio pillow requests

In [ ]:
import os
import base64
from io import BytesIO

import gradio as gr
import requests
from PIL import Image
from openai import OpenAI

In [ ]:
# Set your API key before running the notebook.
# Example in Jupyter:
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
def generate_image(prompt):
    """Generate a design image from a text prompt."""
    prompt = (prompt or "").strip()
    if not prompt:
        raise gr.Error("Please enter a design prompt.")

    try:
        response = client.images.generate(
            model="gpt-image-1",
            prompt=prompt,
            size="1024x1024"
        )

        image_item = response.data[0]
        image_base64 = getattr(image_item, "b64_json", None)
        image_url = getattr(image_item, "url", None)

        if image_base64:
            image_bytes = base64.b64decode(image_base64)
            return Image.open(BytesIO(image_bytes))

        if image_url:
            image_response = requests.get(image_url, timeout=60)
            image_response.raise_for_status()
            return Image.open(BytesIO(image_response.content))

        raise ValueError("The API response did not include image data.")

    except Exception as exc:
        raise gr.Error(f"Image generation failed: {exc}")

In [ ]:
demo = gr.Interface(
    fn=generate_image,
    inputs=gr.Textbox(
        lines=3,
        label="Enter a design prompt",
        placeholder="Example: Create a cinematic Netflix poster for a futuristic sci-fi thriller with neon lights and dramatic shadows"
    ),
    outputs=gr.Image(type="pil", label="Generated Design"),
    title="AI Design Generator for Campaign Visuals",
    description="Enter a prompt to generate custom banner or poster concepts using OpenAI image generation.",
    examples=[
        ["Create a Netflix banner for a fantasy adventure series with a glowing castle, misty forest, and heroic lead character."],
        ["Design a dramatic poster for a crime thriller with rainy city streets, moody lighting, and bold red accents."],
        ["Generate a colorful family movie poster with playful characters, bright lighting, and a cheerful cinematic style."]
    ]
)

demo

In [ ]:
demo.launch()